# OpenPlaque — RCA 10–50 mm PCAT prototype

This notebook starts from `main`, is self-contained, and does **not** rerun centerline extraction.

It consumes the saved `global_graph_centerline_zyx.csv`, loads source CCTA series 7, smooths/resamples the RCA coordinate axis, estimates an approximate local vessel radius from contrast-filled lumen, constructs a perivascular shell for the **10–50 mm RCA segment**, and measures adipose attenuation.

**Important:** this is an OpenPlaque research prototype, not Caristo's proprietary FAI-Score. The true outer vessel wall is not segmented here. We approximate it as the estimated lumen radius plus a configurable wall margin, then sample outward by one estimated local outer diameter.

Default adipose range: **−190 to −30 HU**.


In [ ]:
# FIRST EXECUTABLE CELL: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch pcat-prototype-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy matplotlib pandas

print('Repository and packages ready.')


In [ ]:
import sys, shutil, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy import ndimage as ndi
from scipy.interpolate import splprep, splev
from scipy.spatial import cKDTree

sys.path.insert(0, '/content/OpenPlaque/src')
from openplaque.study import OpenPlaqueStudy

ROOT = Path('/content/drive/MyDrive/OpenPlaque')
OUT = ROOT / 'PCAT_RCA_10_50'
OUT.mkdir(parents=True, exist_ok=True)

FAT_LO_HU = -190.0
FAT_HI_HU = -30.0
SEGMENT_START_MM = 10.0
SEGMENT_END_MM = 50.0
WALL_MARGIN_MM = 0.75
CENTERLINE_STEP_MM = 0.5
RAY_STEP_MM = 0.2
RAY_MAX_MM = 5.0
N_RAYS = 32

print('Output:', OUT)


## 1. Load source CCTA series 7 and the saved global centerline


In [ ]:
DRIVE_ZIP = ROOT / 'Full_DICOM.zip'
LOCAL_ZIP = Path('/content/Full_DICOM.zip')
EXTRACT_ROOT = '/content/full_dicom_pcat'

if not DRIVE_ZIP.exists():
    raise FileNotFoundError(DRIVE_ZIP)
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    print('Copying Full_DICOM.zip locally...')
    shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)

shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
study = OpenPlaqueStudy(str(LOCAL_ZIP), extract_root=EXTRACT_ROOT)
source_img, ct, _ = study.load_series(7)
ct = np.asarray(ct)
sp_xyz = np.array(source_img.GetSpacing(), float)
sp_zyx = sp_xyz[::-1]
voxel_mm3 = float(np.prod(sp_xyz))

CENTERLINE_CANDIDATES = [
    ROOT/'RCA_Global_Graph'/'global_graph_centerline_zyx.csv',
    ROOT/'RCA_Global_Graph_Extension'/'global_graph_centerline_zyx.csv',
    ROOT/'Global_Graph_RCA'/'global_graph_centerline_zyx.csv',
]
centerline_path = next((p for p in CENTERLINE_CANDIDATES if p.exists()), None)
if centerline_path is None:
    # Last-resort recursive search under the OpenPlaque Drive folder.
    hits = list(ROOT.rglob('global_graph_centerline_zyx.csv'))
    centerline_path = hits[0] if hits else None
if centerline_path is None:
    raise FileNotFoundError('Could not find global_graph_centerline_zyx.csv under MyDrive/OpenPlaque.')

cl = pd.read_csv(centerline_path)
required = {'z','y','x'}
if not required.issubset(cl.columns):
    raise ValueError(f'Centerline CSV must contain {required}; got {list(cl.columns)}')
if 'arc_mm' not in cl.columns:
    q = cl[['z','y','x']].to_numpy(float)
    ds = np.linalg.norm(np.diff(q, axis=0)*sp_zyx, axis=1)
    cl['arc_mm'] = np.r_[0, np.cumsum(ds)]

print('CT shape z,y,x:', ct.shape)
print('Spacing x,y,z mm:', tuple(sp_xyz))
print('Centerline:', centerline_path)
print('Centerline length mm:', float(cl.arc_mm.iloc[-1]))
display(cl.head())


## 2. Smooth and resample the RCA coordinate axis

The graph-derived route is useful as a longitudinal coordinate, but its voxelwise path is stair-stepped. We fit a light 3-D smoothing spline and resample it at 0.5 mm spacing. The 0-mm ostium and overall geometry are preserved closely.


In [ ]:
raw = cl[['z','y','x']].to_numpy(float)
raw_mm = raw * sp_zyx
s_raw = cl['arc_mm'].to_numpy(float)

# Parameterize by normalized arc. A small smoothing term removes voxel stair-stepping
# without materially changing the trajectory.
u = (s_raw - s_raw[0]) / max(s_raw[-1] - s_raw[0], 1e-6)
tck, _ = splprep(raw_mm.T, u=u, s=len(raw_mm)*0.05, k=min(3, len(raw_mm)-1))

n = int(np.ceil(s_raw[-1] / CENTERLINE_STEP_MM)) + 1
u_dense = np.linspace(0, 1, n)
smooth_mm = np.column_stack(splev(u_dense, tck))

# Recompute true arc after smoothing and resample uniformly by arc length.
d = np.linalg.norm(np.diff(smooth_mm, axis=0), axis=1)
s_tmp = np.r_[0, np.cumsum(d)]
s_uniform = np.arange(0, s_tmp[-1] + CENTERLINE_STEP_MM*0.5, CENTERLINE_STEP_MM)
smooth_uniform_mm = np.column_stack([
    np.interp(s_uniform, s_tmp, smooth_mm[:,j]) for j in range(3)
])
smooth_uniform = smooth_uniform_mm / sp_zyx

smoothed = pd.DataFrame({
    'z': smooth_uniform[:,0],
    'y': smooth_uniform[:,1],
    'x': smooth_uniform[:,2],
    'arc_mm': s_uniform,
})
smoothed.to_csv(OUT/'rca_centerline_smoothed_zyx.csv', index=False)

print('Smoothed length mm:', round(float(s_uniform[-1]), 2))
print('Points:', len(smoothed))


## 3. Estimate local lumen radius on perpendicular planes

At each centerline point we cast radial rays in the plane perpendicular to the local tangent. The lumen edge is approximated as the first sustained drop below an adaptive contrast threshold. The median across rays is used, then gently smoothed longitudinally.

This estimates **lumen radius**, not the true outer-wall radius.


In [ ]:
def sample_ct(points_zyx):
    p = np.asarray(points_zyx, float)
    coords = np.vstack([p[:,0], p[:,1], p[:,2]])
    return ndi.map_coordinates(ct.astype(np.float32, copy=False), coords, order=1, mode='nearest')

def tangent_at(i):
    a = max(0, i-2)
    b = min(len(smooth_uniform_mm)-1, i+2)
    t = smooth_uniform_mm[b] - smooth_uniform_mm[a]
    t /= max(np.linalg.norm(t), 1e-9)
    return t

def plane_basis(t):
    ref = np.array([1.,0.,0.])
    if abs(np.dot(t, ref)) > 0.85:
        ref = np.array([0.,1.,0.])
    u = np.cross(t, ref)
    u /= max(np.linalg.norm(u), 1e-9)
    v = np.cross(t, u)
    v /= max(np.linalg.norm(v), 1e-9)
    return u, v

rvals = np.arange(0, RAY_MAX_MM + 1e-6, RAY_STEP_MM)
angles = np.linspace(0, 2*np.pi, N_RAYS, endpoint=False)
lumen_r = np.full(len(s_uniform), np.nan, float)
center_hu = sample_ct(smooth_uniform)

for i, (p_mm, ch) in enumerate(zip(smooth_uniform_mm, center_hu)):
    if not (8 <= s_uniform[i] <= 52):
        continue
    t = tangent_at(i)
    u1, u2 = plane_basis(t)
    thr = float(np.clip(0.40*ch, 180, 360))
    ray_est = []
    for a in angles:
        direction_mm = np.cos(a)*u1 + np.sin(a)*u2
        pts_mm = p_mm[None,:] + rvals[:,None]*direction_mm[None,:]
        pts_zyx = pts_mm / sp_zyx
        hu = sample_ct(pts_zyx)
        below = hu < thr
        idx = None
        for j in range(2, len(below)):
            if below[j] and below[j-1]:
                idx = j-1
                break
        if idx is not None:
            ray_est.append(max(0.6, float(rvals[idx])))
    if len(ray_est) >= N_RAYS//3:
        lumen_r[i] = float(np.median(ray_est))

# Fill gaps and constrain to plausible proximal coronary radii.
good = np.isfinite(lumen_r)
if good.sum() < 10:
    raise RuntimeError('Could not estimate enough local lumen radii.')
lumen_r = np.interp(np.arange(len(lumen_r)), np.where(good)[0], lumen_r[good])
lumen_r = np.clip(lumen_r, 0.8, 3.5)
lumen_r = ndi.median_filter(lumen_r, size=7, mode='nearest')
lumen_r = ndi.gaussian_filter1d(lumen_r, sigma=2, mode='nearest')

outer_r = lumen_r + WALL_MARGIN_MM
shell_thickness = 2.0 * outer_r   # one estimated local outer diameter
shell_outer_r = outer_r + shell_thickness

radius_df = pd.DataFrame({
    'arc_mm': s_uniform,
    'center_hu': center_hu,
    'lumen_radius_mm': lumen_r,
    'outer_radius_approx_mm': outer_r,
    'shell_thickness_mm': shell_thickness,
    'shell_outer_radius_mm': shell_outer_r,
})
radius_df.to_csv(OUT/'pcat_local_radius_profile.csv', index=False)

segm = (s_uniform >= SEGMENT_START_MM) & (s_uniform <= SEGMENT_END_MM)
display(radius_df.loc[segm].describe())


## 4. Construct the 10–50 mm perivascular shell

For every voxel in a tight bounding box around the RCA, we find its nearest smoothed centerline point. A voxel belongs to the shell when:

- its nearest centerline arc is 10–50 mm;
- its radial distance is outside the approximate outer wall;
- its radial distance is no more than one local outer diameter beyond that wall.

If the cached TotalSegmentator aorta mask is available, aortic voxels are excluded.


In [ ]:
# Optional aorta exclusion.
aorta = None
aorta_candidates = [
    ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz',
    ROOT/'TotalSegmentator_Validation_v2'/'aorta_series7_totalseg.nii.gz',
]
ap = next((p for p in aorta_candidates if p.exists()), None)
if ap is not None:
    ai = sitk.ReadImage(str(ap))
    if ai.GetSize()!=source_img.GetSize() or not np.allclose(ai.GetSpacing(), source_img.GetSpacing()):
        ai = sitk.Resample(ai, source_img, sitk.Transform(), sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8)
    aorta = sitk.GetArrayFromImage(ai) > 0
    print('Using aorta exclusion:', ap)
else:
    print('Aorta mask not found; continuing without explicit aorta exclusion.')

# Use a slightly padded centerline for nearest-point assignment,
# then analyze only voxels assigned to 10-50 mm.
tree_mask = (s_uniform >= 8.0) & (s_uniform <= 52.0)
tree_pts_mm = smooth_uniform_mm[tree_mask]
tree_arc = s_uniform[tree_mask]
tree_outer_r = outer_r[tree_mask]
tree_shell_outer = shell_outer_r[tree_mask]
tree = cKDTree(tree_pts_mm)

max_shell = float(np.max(tree_shell_outer)) + 1.5
seg_pts = smooth_uniform[(s_uniform>=8)&(s_uniform<=52)]
lo = np.floor(seg_pts.min(axis=0) - max_shell/sp_zyx).astype(int)
hi = np.ceil(seg_pts.max(axis=0) + max_shell/sp_zyx).astype(int) + 1
lo = np.maximum(lo, 0)
hi = np.minimum(hi, np.array(ct.shape))

shape = tuple((hi-lo).astype(int))
shell_crop = np.zeros(shape, np.uint8)
fat_crop = np.zeros(shape, np.uint8)
radial_depth_crop = np.full(shape, np.nan, np.float32)
arc_crop = np.full(shape, np.nan, np.float32)

yy, xx = np.meshgrid(np.arange(lo[1],hi[1]), np.arange(lo[2],hi[2]), indexing='ij')
yx = np.column_stack([yy.ravel(), xx.ravel()])

for z in range(lo[0], hi[0]):
    pts_zyx = np.column_stack([
        np.full(len(yx), z, float),
        yx[:,0].astype(float),
        yx[:,1].astype(float),
    ])
    pts_mm = pts_zyx * sp_zyx
    dist, idx = tree.query(pts_mm, k=1)
    arc_near = tree_arc[idx]
    rin = tree_outer_r[idx]
    rout = tree_shell_outer[idx]
    keep = (
        (arc_near >= SEGMENT_START_MM) &
        (arc_near <= SEGMENT_END_MM) &
        (dist >= rin) &
        (dist <= rout)
    )
    iz = z-lo[0]
    k2 = keep.reshape(shape[1], shape[2])
    shell_crop[iz] = k2.astype(np.uint8)

    hu = ct[z, lo[1]:hi[1], lo[2]:hi[2]]
    fk = k2 & (hu >= FAT_LO_HU) & (hu <= FAT_HI_HU)
    if aorta is not None:
        fk &= ~aorta[z, lo[1]:hi[1], lo[2]:hi[2]]
        shell_crop[iz][aorta[z, lo[1]:hi[1], lo[2]:hi[2]]] = 0
    fat_crop[iz] = fk.astype(np.uint8)

    radial_depth = (dist - rin).reshape(shape[1],shape[2])
    ac = arc_near.reshape(shape[1],shape[2])
    radial_depth_crop[iz][k2] = radial_depth[k2]
    arc_crop[iz][k2] = ac[k2]

print('Crop lo z,y,x:', lo, ' hi:', hi, ' shape:', shape)
print('Shell voxels:', int(shell_crop.sum()))
print('PCAT fat voxels:', int(fat_crop.sum()))


## 5. Compute OpenPlaque PCAT attenuation metrics


In [ ]:
hu_crop = ct[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
fat_hu = hu_crop[fat_crop > 0].astype(float)

if len(fat_hu) < 100:
    raise RuntimeError(f'Too few PCAT adipose voxels ({len(fat_hu)}). Inspect the shell/radius estimates.')

summary = pd.DataFrame([{
    'metric_name': 'OpenPlaque PCAT Attenuation',
    'segment_start_mm': SEGMENT_START_MM,
    'segment_end_mm': SEGMENT_END_MM,
    'fat_hu_low': FAT_LO_HU,
    'fat_hu_high': FAT_HI_HU,
    'pcat_mean_hu': float(np.mean(fat_hu)),
    'pcat_median_hu': float(np.median(fat_hu)),
    'pcat_sd_hu': float(np.std(fat_hu)),
    'pcat_p10_hu': float(np.percentile(fat_hu,10)),
    'pcat_p90_hu': float(np.percentile(fat_hu,90)),
    'fat_voxels': int(len(fat_hu)),
    'fat_volume_ml': float(len(fat_hu)*voxel_mm3/1000),
    'shell_volume_ml': float(shell_crop.sum()*voxel_mm3/1000),
    'fat_fraction_of_shell': float(len(fat_hu)/max(shell_crop.sum(),1)),
    'mean_lumen_radius_mm': float(np.mean(lumen_r[segm])),
    'mean_outer_radius_approx_mm': float(np.mean(outer_r[segm])),
    'mean_shell_thickness_mm': float(np.mean(shell_thickness[segm])),
    'outer_wall_method': f'lumen radius + {WALL_MARGIN_MM:.2f} mm approximation',
}])
summary.to_csv(OUT/'pcat_summary.csv', index=False)
display(summary.T)


## 6. Longitudinal and radial attenuation profiles


In [ ]:
# Longitudinal 1-mm bins.
long_rows = []
for a0 in np.arange(SEGMENT_START_MM, SEGMENT_END_MM, 1.0):
    m = (fat_crop>0) & (arc_crop>=a0) & (arc_crop<a0+1)
    vals = hu_crop[m].astype(float)
    long_rows.append({
        'arc_start_mm': a0,
        'arc_mid_mm': a0+0.5,
        'n_voxels': int(len(vals)),
        'mean_hu': float(np.mean(vals)) if len(vals) else np.nan,
        'median_hu': float(np.median(vals)) if len(vals) else np.nan,
    })
long_df = pd.DataFrame(long_rows)
long_df.to_csv(OUT/'pcat_longitudinal_profile.csv', index=False)

# Absolute radial 1-mm layers outward from approximate outer wall.
max_depth = float(np.nanmax(radial_depth_crop[fat_crop>0]))
rad_rows=[]
for r0 in np.arange(0, math.ceil(max_depth), 1.0):
    m = (fat_crop>0) & (radial_depth_crop>=r0) & (radial_depth_crop<r0+1)
    vals = hu_crop[m].astype(float)
    if len(vals):
        rad_rows.append({
            'radial_start_mm': r0,
            'radial_mid_mm': r0+0.5,
            'n_voxels': int(len(vals)),
            'mean_hu': float(np.mean(vals)),
            'median_hu': float(np.median(vals)),
        })
rad_df=pd.DataFrame(rad_rows)
rad_df.to_csv(OUT/'pcat_radial_profile.csv', index=False)

display(long_df.head(10))
display(rad_df)


## 7. Save NIfTI shell and PCAT-fat masks


In [ ]:
def save_full_mask(crop, path):
    full = np.zeros(ct.shape, np.uint8)
    full[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]] = crop.astype(np.uint8)
    im = sitk.GetImageFromArray(full)
    im.CopyInformation(source_img)
    sitk.WriteImage(im, str(path))
    del full, im

save_full_mask(shell_crop, OUT/'rca_10_50_pcat_shell_mask.nii.gz')
save_full_mask(fat_crop, OUT/'rca_10_50_pcat_fat_mask.nii.gz')

print('Saved NIfTI masks.')


## 8. Visual QC


In [ ]:
# Overview: 10,20,30,40,50 mm axial slices with shell/fat overlays.
targets = [10,20,30,40,50]
fig, axs = plt.subplots(1,5, figsize=(20,4))
for ax, target in zip(axs, targets):
    j = int(np.argmin(np.abs(s_uniform-target)))
    z,y,x = np.round(smooth_uniform[j]).astype(int)
    rpx = int(np.ceil(18.0/sp_zyx[1]))
    y0=max(0,y-rpx); y1=min(ct.shape[1],y+rpx+1)
    x0=max(0,x-rpx); x1=min(ct.shape[2],x+rpx+1)

    ax.imshow(ct[z,y0:y1,x0:x1], cmap='gray', vmin=-200, vmax=800)
    if lo[0] <= z < hi[0]:
        iz=z-lo[0]
        sy0=max(y0,lo[1])-lo[1]; sy1=min(y1,hi[1])-lo[1]
        sx0=max(x0,lo[2])-lo[2]; sx1=min(x1,hi[2])-lo[2]
        # Map crop contours only when crop overlaps displayed region.
        yy0=max(y0,lo[1]); yy1=min(y1,hi[1])
        xx0=max(x0,lo[2]); xx1=min(x1,hi[2])
        if yy1>yy0 and xx1>xx0:
            shell2=np.zeros((y1-y0,x1-x0),bool)
            fat2=np.zeros_like(shell2)
            shell2[yy0-y0:yy1-y0,xx0-x0:xx1-x0] = shell_crop[iz,yy0-lo[1]:yy1-lo[1],xx0-lo[2]:xx1-lo[2]]>0
            fat2[yy0-y0:yy1-y0,xx0-x0:xx1-x0] = fat_crop[iz,yy0-lo[1]:yy1-lo[1],xx0-lo[2]:xx1-lo[2]]>0
            if shell2.any(): ax.contour(shell2.astype(float), levels=[.5], linewidths=1)
            yyf,xxf=np.where(fat2)
            if len(xxf): ax.scatter(xxf,yyf,s=2,alpha=.25)
    ax.plot(x-x0,y-y0,'o',markersize=4)
    ax.set_title(f'{target} mm, z={z}')
    ax.axis('off')
plt.tight_layout()
p1=OUT/'01_pcat_axial_qc.png'
fig.savefig(p1,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)

# Profiles.
fig,axs=plt.subplots(1,3,figsize=(17,4))
axs[0].plot(radius_df.loc[segm,'arc_mm'],radius_df.loc[segm,'lumen_radius_mm'],label='lumen radius')
axs[0].plot(radius_df.loc[segm,'arc_mm'],radius_df.loc[segm,'outer_radius_approx_mm'],label='outer radius approx')
axs[0].set_xlabel('arc from ostium (mm)'); axs[0].set_ylabel('radius (mm)'); axs[0].legend(); axs[0].set_title('radius model')

axs[1].plot(long_df.arc_mid_mm,long_df.mean_hu)
axs[1].axhline(summary.iloc[0].pcat_mean_hu,linestyle='--')
axs[1].set_xlabel('arc from ostium (mm)'); axs[1].set_ylabel('PCAT mean HU'); axs[1].set_title('longitudinal PCAT')

axs[2].plot(rad_df.radial_mid_mm,rad_df.mean_hu,marker='o')
axs[2].set_xlabel('mm outward from approx outer wall'); axs[2].set_ylabel('PCAT mean HU'); axs[2].set_title('radial PCAT profile')

plt.tight_layout()
p2=OUT/'02_pcat_profiles.png'
fig.savefig(p2,dpi=180,bbox_inches='tight')
plt.show(); plt.close(fig)

print('Saved:', p1)
print('Saved:', p2)


## 9. Interpretation guardrails

This prototype reports **OpenPlaque PCAT Attenuation**, not FAI-Score.

The key approximation is the vessel outer wall. The present notebook estimates contrast-filled lumen radius from perpendicular ray sampling and then adds a fixed 0.75 mm wall margin. A future version should replace this approximation with a validated coronary outer-wall segmentation if one becomes available.

For the current experiment, the main questions are:

1. Does the 10–50 mm shell visually follow the RCA?
2. Does the PCAT-fat mask stay in perivascular adipose rather than myocardium/lung?
3. Is the mean attenuation stable enough longitudinally and radially to justify further validation?


In [ ]:
print('PCAT prototype complete.')
print('Please send:')
print('  01_pcat_axial_qc.png')
print('  02_pcat_profiles.png')
print('  pcat_summary.csv row')
